In [1]:
# Cell 1 - Install & Import
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
from sklearn.metrics import average_precision_score
import xgboost as xgb
import lightgbm as lgb

print("All imports done!")

All imports done!


In [2]:
# Cell 2 - Load data from GitHub
url = "https://raw.githubusercontent.com/Shar1q-codes/factoryguard-ai/master/data/train_FD001_with_RUL.csv"

df = pd.read_csv(url)
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print(df.head(2))

HTTPError: HTTP Error 404: Not Found

In [3]:
from google.colab import files
uploaded = files.upload()

Saving train_FD001_with_RUL.csv to train_FD001_with_RUL.csv


In [4]:
df = pd.read_csv('train_FD001_with_RUL.csv')
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

Shape: (24640, 29)
Columns: ['unit_nr', 'time_cycles', 'op_setting_1', 'op_setting_2', 'op_setting_3', 's1', 's2', 's3', 's4', 's5', 's6', 's7', 's8', 's9', 's10', 's11', 's12', 's13', 's14', 's15', 's16', 's17', 's18', 's19', 's20', 's21', 'max_cycles', 'RUL', 'failure']


In [5]:
sensors = [f's{i}' for i in range(1, 22)]
print("Sensors:", sensors)

Sensors: ['s1', 's2', 's3', 's4', 's5', 's6', 's7', 's8', 's9', 's10', 's11', 's12', 's13', 's14', 's15', 's16', 's17', 's18', 's19', 's20', 's21']


In [6]:
def add_rolling_features(df, sensors, windows=[5, 10, 20]):
    for col in sensors:
        for w in windows:
            df[f'{col}_roll_mean_{w}'] = df.groupby('unit_nr')[col].transform(
                lambda x: x.rolling(w, min_periods=1).mean()
            )
            df[f'{col}_roll_std_{w}'] = df.groupby('unit_nr')[col].transform(
                lambda x: x.rolling(w, min_periods=1).std().fillna(0)
            )
            df[f'{col}_ema_{w}'] = df.groupby('unit_nr')[col].transform(
                lambda x: x.ewm(span=w, adjust=False).mean()
            )
    return df

df = add_rolling_features(df, sensors)
df = df.copy()
print("Rolling features done! Shape:", df.shape)

/tmp/ipykernel_10231/3453699609.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'{col}_ema_{w}'] = df.groupby('unit_nr')[col].transform(
/tmp/ipykernel_10231/3453699609.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'{col}_roll_mean_{w}'] = df.groupby('unit_nr')[col].transform(
/tmp/ipykernel_10231/3453699609.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat

Rolling features done! Shape: (24640, 218)


/tmp/ipykernel_10231/3453699609.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'{col}_roll_std_{w}'] = df.groupby('unit_nr')[col].transform(
/tmp/ipykernel_10231/3453699609.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'{col}_ema_{w}'] = df.groupby('unit_nr')[col].transform(
/tmp/ipykernel_10231/3453699609.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(

In [7]:
df = df.copy()
print("Rolling features done! Shape:", df.shape)

Rolling features done! Shape: (24640, 218)


In [8]:
def add_lag_features(df, sensors):
    for col in sensors:
        df[f'{col}_lag1'] = df.groupby('unit_nr')[col].transform(
            lambda x: x.shift(1)
        ).fillna(0)
        df[f'{col}_lag2'] = df.groupby('unit_nr')[col].transform(
            lambda x: x.shift(2)
        ).fillna(0)
    return df

df = add_lag_features(df, sensors)
df = df.copy()
print("Lag features done! Shape:", df.shape)

Lag features done! Shape: (24640, 260)


In [9]:
drop_cols = ['unit_nr', 'time_cycles', 'max_cycles', 'RUL']

units = df['unit_nr'].unique()
train_units = units[:int(len(units)*0.8)]
test_units = units[int(len(units)*0.8):]

train = df[df['unit_nr'].isin(train_units)]
test = df[df['unit_nr'].isin(test_units)]

X_train = train.drop(columns=drop_cols)
y_train = train['failure']
X_test = test.drop(columns=drop_cols)
y_test = test['failure']

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

X_train shape: (19468, 256)
X_test shape: (5172, 256)


In [10]:
import xgboost as xgb
import lightgbm as lgb
from sklearn.metrics import average_precision_score

ratio = (y_train == 0).sum() / (y_train == 1).sum()

# XGBoost
xgb_model = xgb.XGBClassifier(
    scale_pos_weight=ratio,
    n_estimators=200,
    eval_metric='aucpr'
)
xgb_model.fit(X_train, y_train)
xgb_prauc = average_precision_score(y_test, xgb_model.predict_proba(X_test)[:,1])
print(f"XGBoost PR-AUC: {xgb_prauc:.4f}")

# LightGBM
lgb_model = lgb.LGBMClassifier(scale_pos_weight=ratio, n_estimators=200)
lgb_model.fit(X_train, y_train)
lgb_prauc = average_precision_score(y_test, lgb_model.predict_proba(X_test)[:,1])
print(f"LightGBM PR-AUC: {lgb_prauc:.4f}")

# Save compressed
import joblib
joblib.dump(df, 'features_df.pkl', compress=3)
print("Saved features_df.pkl!")

from google.colab import files
files.download('features_df.pkl')

XGBoost PR-AUC: 1.0000
[LightGBM] [Info] Number of positive: 2480, number of negative: 16988
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.073636 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 65027
[LightGBM] [Info] Number of data points in the train set: 19468, number of used features: 256
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.127389 -> initscore=-1.924249
[LightGBM] [Info] Start training from score -1.924249
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>